# 01 — Your First Agent

**Steer** — you decide what the agent tries to do, with a plain-English `instructions` string.

When you finish you can:

- Build an `Agent` from a model string and `instructions`, and run it
- Read the answer (`result.output`) and the bill (`result.usage`)
- See that a run is just a list of messages going in and coming out

In [ ]:
from learningdantic import setup

MODEL = setup()

## The idea

An agent is **a model + instructions + tools + an output type**. This notebook uses the first two:

1. **The model**: a string with a provider prefix, like your `MODEL` (`'openai:…'`). A name without the
   prefix raises `UserError`.
2. **The instructions**: what the agent should do, in plain English. This is your first way to **steer** it.
   (Older code uses `system_prompt=`. It still works, but `instructions=` is preferred.)

Output types and tools come in notebooks 03 and 04.

One line about `await`: it tells the notebook to wait for the model's answer. Put it in front of
`agent.run(...)`, and that is all you need to know about async here.

In [ ]:
from pydantic_ai import Agent

tutor = Agent(
    MODEL,
    instructions="You are a patient programming tutor. Answer with exactly one clear sentence.",
)

result = await tutor.run("What is an AI agent, really?")
print(result.output)

> ✅ **Checkpoint** — one clear sentence about AI agents. You just paid for it (a tiny fraction of a cent).
>
> 🔭 **In Logfire** (optional, if you set it up in notebook 00): open the **Live view** at
> [logfire.pydantic.dev](https://logfire.pydantic.dev), find this run and expand it. The model call shows a
> coin badge with its tokens and cost.

## Read the result

`result.output` is the answer. `result.usage` is the bill: tokens in, tokens out, and the cost in US dollars.
`usage` is a property, so there are no parentheses: `result.usage`, not `result.usage()`.

(Copied code with `result.data` or `result.usage()` from an older tutorial? See
[Coming from an old tutorial?](../README.md#coming-from-an-old-tutorial) in the README.)

In [ ]:
print("output       :", result.output)
print("usage        :", result.usage)
print("input tokens :", result.usage.input_tokens)
print("output tokens:", result.usage.output_tokens)
print("cost (USD)   :", result.usage.cost)

> ✅ **Checkpoint** — something like `cost (USD)   : 0.0000422`. That's your bill in US dollars (a `Decimal`).
> If it prints `None`, pydantic-ai doesn't know your model's price yet; the tokens are still counted.
> On a reasoning model, `output_tokens` also counts the tokens it spent thinking, which is why it's bigger
> than one sentence. The thinking share is in `result.usage.details`, under the provider's own name
> (`reasoning_tokens` on OpenAI). pydantic-ai also copies it to `result.usage.output_reasoning_tokens`, but
> only for providers its price data knows, so that one isn't always there.

## What just happened, message by message

An agent run is just a list of **messages**. The model has no hidden memory.
`result.all_messages()` gives you the whole exchange, which is exactly what you'll feed
back in notebook 07 to give an agent "memory".

In [ ]:
for message in result.all_messages():
    print(type(message).__name__, "→", [type(part).__name__ for part in message.parts])

> ✅ **Checkpoint** — something like:
>
> ```
> ModelRequest → ['UserPromptPart']
> ModelResponse → ['ThinkingPart', 'TextPart']
> ```
>
> The request holds your prompt; the response holds the answer (`TextPart`). A reasoning model adds a
> `ThinkingPart` first; other models skip it. Your instructions aren't a part: they travel with the request.
> Messages in, messages out: that's the whole mental model.

## 🏋️ Exercise — the pirate agent

Build `pirate_agent` so that it answers **any** question in full pirate dialect ("Arrr, the answer be...").

1. Run it on a question of your choice and print `result.output`.
2. Print `result.usage.output_tokens`: how many tokens did the dialect cost you?

Stretch: run the same question twice. Do you get the same answer? Compare the token counts.

Try it in a new cell before you open the solution:
[`../solutions/01_pirate_agent.ipynb`](../solutions/01_pirate_agent.ipynb).

## Next →

**[02_models_and_providers.ipynb](02_models_and_providers.ipynb)** — model strings, settings, fallbacks, and what
different models charge you.